# Saber BERM Scope Sweep — GSM8K (LIMIT=150)

探索 BERM 作用范围对精度/NFE 的影响。固定 `saber_expand=True, berm_mode=cross_step, saber_global_aadu=True, saber_mtr=0.8`。

| # | 方法 | berm_scope | 说明 |
|---|------|------------|------|
| 1 | baseline | `window` | 原始行为：BERM 作用于整个扩展窗口（含所有已 Expand 的 block） |
| 2 | current_block | `current_block` | BERM 只在最新 watching block 内操作；全量 forward 不触发 BERM |
| 3 | fullforward_berm | `fullforward_berm` | 同上，但 rewarm 时允许全窗口 BERM（可退回之前 block） |
| 4 | trail_block | `trail_block` | BERM 在 watching_nb-1 到 e 范围内（当前 + 前一个 block），再之前的不动 |

共 18 个任务。baseline(window) 3个 + current_block 6个 + fullforward_berm 3个 + trail_block 6个。全部 bl=32。

## 1. 环境设置

In [ ]:
import os, gc, re, json, datetime, threading, queue, subprocess
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '4,5')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)
os.makedirs('evals_results/saber', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

## 2. 任务配置

In [ ]:
task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 256
steps = 256
limit_samples = 150
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def _task(name, n, mu, berm_scope='window'):
    extra_args = [
        'saber_expand=True',
        'berm_mode=cross_step',
        'saber_global_aadu=True',
        'saber_mtr=0.8',
        f'saber_n={n}',
        f'saber_mu={mu}',
        'block_length=32',
        f'saber_berm_scope={berm_scope}',
    ]
    return {'name': name, 'extra_args': extra_args}


TASK_CONFIGS = [
    # anchor: 现有最强配置（window scope, 全量 BERM 窗口）
    _task('saber_exp_gl_cs_n4_mu8_bl32_base', 4, 8, 'window'),

    # 1) current_block: BERM only in latest watching block, no BERM on rewarm
    _task('bs_curblk_n3_mu8', 3, 8, 'current_block'),
    _task('bs_curblk_n4_mu8', 4, 8, 'current_block'),
    _task('bs_curblk_n5_mu8', 5, 8, 'current_block'),
    _task('bs_curblk_n6_mu8', 6, 8, 'current_block'),
    _task('bs_curblk_n8_mu8', 8, 8, 'current_block'),
    _task('bs_curblk_n4_mu4', 4, 4, 'current_block'),
    _task('bs_curblk_n4_mu6', 4, 6, 'current_block'),
    _task('bs_curblk_n4_mu12', 4, 12, 'current_block'),

    # 2) fullforward_berm: same as current_block + BERM on rewarm
    _task('bs_ffberm_n4_mu8', 4, 8, 'fullforward_berm'),
    _task('bs_ffberm_n5_mu8', 5, 8, 'fullforward_berm'),
    _task('bs_ffberm_n6_mu8', 6, 8, 'fullforward_berm'),
    _task('bs_ffberm_n4_mu4', 4, 4, 'fullforward_berm'),
    _task('bs_ffberm_n4_mu12', 4, 12, 'fullforward_berm'),

    # 3) trail_block: BERM on current + one trailing block
    _task('bs_trail_n3_mu8', 3, 8, 'trail_block'),
    _task('bs_trail_n4_mu8', 4, 8, 'trail_block'),
    _task('bs_trail_n5_mu8', 5, 8, 'trail_block'),
    _task('bs_trail_n6_mu8', 6, 8, 'trail_block'),
    _task('bs_trail_n8_mu8', 8, 8, 'trail_block'),
    _task('bs_trail_n4_mu4', 4, 4, 'trail_block'),
    _task('bs_trail_n4_mu6', 4, 6, 'trail_block'),
    _task('bs_trail_n4_mu12', 4, 12, 'trail_block'),
    _task('bs_trail_n6_mu4', 6, 4, 'trail_block'),
    _task('bs_trail_n6_mu12', 6, 12, 'trail_block'),
]

GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
print(f'Total tasks: {len(TASK_CONFIGS)} | GPUs: {GPU_POOL} | limit: {limit_samples} | timestamp: {timestamp}')

## 3. 并行启动任务

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_bermscope_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

        common_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            'show_speed=True',
            f'seed={seed}',
        ]
        model_args = ','.join(common_args + cfg['extra_args'])
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} --confirm_run_unsafe_code '
            f'--model llada_dist --model_args {model_args} --output_path {output_dir} '
            f'--log_samples --limit {limit_samples}'
        )

        print(f'[GPU {gpu_id}] START {name}')
        p = subprocess.Popen(
            cmd,
            shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()
        print(f'[GPU {gpu_id}] DONE  {name} rc={rc}')
        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))
        task_queue.task_done()


threads = []
for gid in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print('All finished:', len(all_results), '/', len(TASK_CONFIGS))

## 4. 解析评测结果

In [ ]:
def parse_result(cfg, output_dir, log_file):
    name = cfg['name']

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    scope_m = re.search(r'bs_(\w+?)_n', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    return {
        'name': name,
        'scope': scope_m.group(1) if scope_m else '?',
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


rows = []
for cfg, name, log_file, output_dir, rc in all_results:
    rows.append(parse_result(cfg, output_dir, log_file))

df = pd.DataFrame(rows).sort_values(['n', 'scope'])
pd.set_option('display.max_rows', 50)
display(df)

print('\nTop by FlexAcc:')
display(df.sort_values('flex_acc', ascending=False))

## 5. 对比图

In [ ]:
plot_df = df[df['flex_acc'].notna()].copy()
plot_df['label'] = plot_df['name']

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
metrics = [('flex_acc', 'FlexAcc'), ('total_nfe', 'Total NFE'), ('tok_per_sec', 'Tokens/sec')]

scope_colors = {'window': '#E53935', 'curblk': '#4E79A7', 'ffberm': '#59A14F', 'trail': '#EDC948'}
colors = [scope_colors.get(r['scope'], '#999') for _, r in plot_df.iterrows()]

for ax, (col, title) in zip(axes, metrics):
    vals = plot_df[col].fillna(0)
    ax.bar(plot_df['label'], vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].tick_params(axis='x', rotation=45, labelsize=9)
plt.tight_layout()
plt.show()

## 6. 从已有结果重新加载（可选）

内核重启后运行此 cell，再跑 Section 5 出图。

In [ ]:
import glob, re, json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

task = 'gsm8k'
seed = 42

TASK_CONFIGS = [
    {'name': 'saber_exp_gl_cs_n4_mu8_bl32_base'},
    {'name': 'bs_curblk_n3_mu8'},
    {'name': 'bs_curblk_n4_mu8'},
    {'name': 'bs_curblk_n5_mu8'},
    {'name': 'bs_curblk_n6_mu8'},
    {'name': 'bs_curblk_n8_mu8'},
    {'name': 'bs_curblk_n4_mu4'},
    {'name': 'bs_curblk_n4_mu6'},
    {'name': 'bs_curblk_n4_mu12'},
    {'name': 'bs_ffberm_n4_mu8'},
    {'name': 'bs_ffberm_n5_mu8'},
    {'name': 'bs_ffberm_n6_mu8'},
    {'name': 'bs_ffberm_n4_mu4'},
    {'name': 'bs_ffberm_n4_mu12'},
    {'name': 'bs_trail_n3_mu8'},
    {'name': 'bs_trail_n4_mu8'},
    {'name': 'bs_trail_n5_mu8'},
    {'name': 'bs_trail_n6_mu8'},
    {'name': 'bs_trail_n8_mu8'},
    {'name': 'bs_trail_n4_mu4'},
    {'name': 'bs_trail_n4_mu6'},
    {'name': 'bs_trail_n4_mu12'},
    {'name': 'bs_trail_n6_mu4'},
    {'name': 'bs_trail_n6_mu12'},
]

first_name = TASK_CONFIGS[0]['name']
latest_logs = sorted(
    glob.glob(f'nlogs/sweep_bermscope_{task}_{first_name}_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest_logs:
    fname = os.path.basename(latest_logs[0])
    timestamp = fname.replace(f'sweep_bermscope_{task}_{first_name}_', '').replace('.log', '')
    print(f'Auto-detected latest timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

parsed_results = []
for cfg in TASK_CONFIGS:
    name = cfg['name']
    log_file = f'nlogs/sweep_bermscope_{task}_{name}_{timestamp}.log'
    output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

    if not os.path.exists(log_file):
        print(f'  [MISS] {name}')
        continue

    with open(log_file, 'r') as f:
        content = f.read()

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    if flex_acc is None:
        m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
        if m: flex_acc = float(m.group(1))
    if strict_acc is None:
        m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
        if m: strict_acc = float(m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', content)
    nfe_m = re.search(r'Total NFE is (\d+)', content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', content)

    scope_m = re.search(r'bs_(\w+?)_n', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)

    parsed_results.append({
        'name': name,
        'scope': scope_m.group(1) if scope_m else '?',
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })
    print(f'  [OK]   {name}')

df = pd.DataFrame(parsed_results).sort_values(['n', 'scope'])

print(f'\nLoaded {len(parsed_results)} results (seed={seed}, timestamp={timestamp})')
print(f'\n{"Name":<26} {"Scope":<10} {"n":<4} {"mu":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 94)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<26} {r['scope']:<10} {r['n']:<4} {r['mu']:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

print(f'\nRe-run Section 5 to regenerate plots.')